# 第3章 物の位置を座標で取る (2D グラウンディングとポインティング)

Cosmos 3 の Reasoner は、文章だけでなく**画像の中の座標**も答えられます。

- **枠 (bounding box)**: `{"bbox_2d": [x1, y1, x2, y2], "label": "名前"}`
- **点**: `{"point_2d": [x, y], "label": "名前"}`

座標は**左上が原点、右が x、下が y**で、**縦横それぞれ 0〜1000 に正規化**されています (画像の縦横比とは関係なく、各軸を別々に 0〜1000 にする)。
ピクセルに直すときは `x / 1000 * 幅`、`y / 1000 * 高さ` です。

この章で試すこと:

1. 1 つの物を枠で囲ませる
2. 複数の物をまとめて枠で囲ませる
3. 点で指させる (ロボットがつかむ場所など)
4. 印を付けた物を説明させる (Describe Anything)
5. 1 → 4 をつなげる: 見つけた物に番号を振って、1 つずつ説明させる
6. 自分の画像で試す

## 準備

**ランタイム**: 「ランタイム → ランタイムのタイプを変更」で **L4** か **A100** の GPU を選んでください (T4 は対象外。Ampere 以降の GPU と bf16 が前提です)。
Reasoner (文章で答える部分) だけを使うので、GPU メモリは 10GB 前後で足ります。

上から順に実行してください。「モデルを読み込む」セルは初回に 5GB ほどダウンロードします。

In [ ]:
# @title GPU を確かめる
import torch
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB")
    assert torch.cuda.get_device_capability()[0] >= 8, "Ampere 以降の GPU (L4 / A100 など) を選んでください。T4 では動きません。"
else:
    print("GPU が見つかりません。Colab ならランタイムのタイプで L4 か A100 を選んでください。")

In [ ]:
# @title インストール (1〜2 分)
!pip install -q -U "transformers>=5.18" accelerate av huggingface_hub

In [ ]:
# @title サンプル素材を取ってくる関数
import os
import urllib.request
from pathlib import Path

ASSETS = Path(os.environ.get("COSMOS_ASSETS", "assets"))
ASSETS.mkdir(parents=True, exist_ok=True)
# NVIDIA の公式クックブック (cosmos-framework) とモデルカードに置かれているお手本の素材
FW = "https://media.githubusercontent.com/media/NVIDIA/cosmos-framework/main/.agents/skills/cookbooks/cosmos3"
HF = "https://huggingface.co/nvidia/Cosmos3-Edge/resolve/main/assets"


def fetch(url):
    """URL の素材を assets/ に落として、そのパスを返す (2 回目からは落とさない)"""
    path = ASSETS / url.rsplit("/", 1)[-1]
    if not path.exists():
        urllib.request.urlretrieve(url, path)
    return str(path)

In [ ]:
# @title モデルを読み込む (初回は 5GB ほどダウンロード)
import json
import re

from IPython.display import Image as IPImage, Video, display
from PIL import Image, ImageDraw, ImageOps
from transformers import AutoModelForImageTextToText, AutoProcessor

MODEL_ID = "nvidia/Cosmos3-Edge"
# Transformers は統合チェックポイントから Reasoner の部分だけを読み込む (生成用の部分は捨てる)
model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, dtype=torch.bfloat16, device_map="auto")
processor = AutoProcessor.from_pretrained(MODEL_ID)

In [ ]:
# @title 質問用の関数を用意する
import json
import re

import av
import numpy as np
from PIL import Image, ImageDraw, ImageOps
from transformers.video_utils import VideoMetadata

VIDEO_EXT = (".mp4", ".mov", ".webm", ".avi", ".mkv")


def load_image(path, max_side=1280):
    """画像を RGB で読む。大きすぎる画像はトークンが増えて遅くなるので縮める (座標は 0〜1000 の比率なので影響しない)"""
    img = path if isinstance(path, Image.Image) else ImageOps.exif_transpose(Image.open(path))
    img = img.convert("RGB")
    img.thumbnail((max_side, max_side))
    return img


def read_video(path, fps=4, max_side=640, max_frames=256):
    """動画を fps 枚/秒に間引いて読む。返すのはコマの配列と、各コマの時刻を伝えるための情報 (VideoMetadata)。
    Reasoner は各コマに「<1.5 seconds>」のような時刻を付けて読むので、時刻を答える問題 (第4章) にはこの情報が欠かせない"""
    frames, t_next = [], 0.0
    with av.open(path) as c:
        for f in c.decode(video=0):
            if f.time is None or f.time + 1e-6 < t_next:
                continue
            img = f.to_image()
            img.thumbnail((max_side, max_side))  # 1 コマのトークン数を抑える
            frames.append(np.asarray(img.convert("RGB")))
            t_next += 1 / fps
            if len(frames) >= max_frames:
                break
    n = len(frames)
    return np.stack(frames), VideoMetadata(total_num_frames=n, fps=fps, frames_indices=list(range(n)), duration=n / fps)


def ask(prompt, media=None, thinking=False, max_new_tokens=2048, fps=4, seed=0):
    """画像・動画 (なくてもよい) と質問を渡して、答えの文字列を返す。

    thinking=True のときは考えた過程 (<think>…</think>) を出してから答える。考えた過程は ask.last_think に残る。
    動画は fps 枚/秒で間引いて読む (推奨は 4)。
    """
    media = [] if media is None else media if isinstance(media, (list, tuple)) else [media]
    content, metadata = [], []
    for m in media:  # 画像・動画は質問文より前に置く (公式の推奨)
        if isinstance(m, str) and m.lower().endswith(VIDEO_EXT):
            frames, meta = read_video(m, fps=fps)
            content.append({"type": "video", "video": frames})
            metadata.append(meta)
        else:
            content.append({"type": "image", "image": load_image(m)})
    content.append({"type": "text", "text": prompt})
    # 動画は read_video で間引き済みなので、ここではもう間引かない
    video_kwargs = {"video_metadata": metadata, "do_sample_frames": False} if metadata else {}
    inputs = processor.apply_chat_template(
        [{"role": "user", "content": content}],
        tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt",
        enable_thinking=thinking, **video_kwargs,
    ).to(model.device)
    # 公式のプロンプトガイドのサンプリング設定 (考える / 考えない で値が違う)
    sampling = dict(temperature=0.6, top_p=0.95, top_k=20) if thinking else dict(temperature=0.7, top_p=0.8, top_k=20)
    torch.manual_seed(seed)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, **sampling)
    text = processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    ask.last_think = ""
    if "</think>" in text:
        think, text = text.split("</think>", 1)
        ask.last_think = think.replace("<think>", "").strip()
    return text.strip()


def parse_json(text):
    """答えの中の JSON (```json … ``` で囲まれていてもよい) を取り出す。読めなければ None。
    小さなモデルの答えは、最後に余計な括弧が付く・真偽値の後ろに " が付く・最後に余計なカンマがある、
    といった崩れ方をするので、括弧の対応を数えて最初のまとまりだけを取り出し、よくある崩れを直してから読む"""
    m = re.search(r"```(?:json)?\s*(.*?)```", text, re.S)
    body = m.group(1) if m else text
    starts = [i for i in (body.find("["), body.find("{")) if i >= 0]
    if not starts:
        return None
    start = min(starts)
    depth, in_str, esc, end = 0, False, False, len(body)
    for k in range(start, len(body)):
        ch = body[k]
        if in_str:
            esc = (ch == "\\") and not esc
            if ch == '"' and not esc:
                in_str = False
            continue
        if ch == '"':
            in_str = True
        elif ch in "[{":
            depth += 1
        elif ch in "]}":
            depth -= 1
            if depth == 0:
                end = k + 1
                break
    chunk = body[start:end]
    fixed = re.sub(r'\b(true|false|null)"', r"\1", re.sub(r",\s*([\]}])", r"\1", chunk))
    for attempt in (chunk, fixed):
        try:
            return json.loads(attempt)
        except json.JSONDecodeError:
            pass
    print("(JSON として読めませんでした)")
    return None


def load_reasoner():
    """Reasoner を読み込む (生成の章で、Generator と入れ替えて使うとき用)"""
    global model, processor
    from transformers import AutoModelForImageTextToText, AutoProcessor
    model = AutoModelForImageTextToText.from_pretrained("nvidia/Cosmos3-Edge", dtype=torch.bfloat16, device_map="auto")
    processor = AutoProcessor.from_pretrained("nvidia/Cosmos3-Edge")

In [ ]:
# @title 座標を画像に描く関数
from PIL import ImageFont

COLORS = ["#e6194b", "#3cb44b", "#4363d8", "#f58231", "#911eb4", "#42d4f4", "#f032e6", "#bfef45"]


def to_px(x, y, img):
    """Cosmos の座標 (左上が原点、x・y それぞれ 0〜1000 に正規化) をピクセルに直す"""
    return x / 1000 * img.width, y / 1000 * img.height


def draw(img, items, path=False, width=3):
    """bbox_2d (枠) と point_2d (点) を描いた画像を返す。path=True なら点を順に線でつなぐ (軌跡)"""
    img = load_image(img).copy()
    d = ImageDraw.Draw(img)
    font = ImageFont.load_default(size=max(14, img.width // 45))
    items = items if isinstance(items, list) else [items]
    pts = []
    for i, it in enumerate(items):
        c = COLORS[i % len(COLORS)]
        label = str(it.get("label", i))
        if "bbox_2d" in it:
            x1, y1 = to_px(*it["bbox_2d"][:2], img)
            x2, y2 = to_px(*it["bbox_2d"][2:4], img)
            d.rectangle([x1, y1, x2, y2], outline=c, width=width)
            d.text((x1 + 4, y1 + 2), label, fill=c, font=font, stroke_width=2, stroke_fill="white")
        if "point_2d" in it:
            x, y = to_px(*it["point_2d"][:2], img)
            pts.append((x, y, label))
    if path and len(pts) > 1:
        d.line([(x, y) for x, y, _ in pts], fill="#ffd400", width=width + 1)
    r = max(5, img.width // 110)
    for j, (x, y, label) in enumerate(pts):
        c = "#ffd400" if path else COLORS[j % len(COLORS)]
        d.ellipse([x - r, y - r, x + r, y + r], fill=c, outline="black", width=2)
        text = str(j + 1) if path else label
        d.text((x + r + 3, y - r), text, fill=c, font=font, stroke_width=2, stroke_fill="black")
    return img

## 1. 1 つの物を枠で囲ませる

公式のプロンプトガイドのお手本です。「Return a json.」と付けると `bbox_2d` の形で返します。

In [ ]:
img = fetch(f"{FW}/reasoner/assets/grounding_2d.png")
answer = ask("Locate the accurate bounding box of the load as a whole. Return a json.", img)
print(answer)
items = parse_json(answer)
display(draw(img, items))

## 2. 複数の物をまとめて枠で囲ませる

欲しい形を例で示すと、その形のリストで返してきます。

In [ ]:
img = fetch(f"{HF}/example_reasoning_input.png")
answer = ask("""Locate every object on the table and output their bounding boxes.
Return a JSON list like this: [{"bbox_2d": [x1, y1, x2, y2], "label": "object name"}]""", img)
print(answer)
boxes = parse_json(answer) or []
display(draw(img, boxes))

## 3. 点で指させる

ロボットに「どこをつかむか」「どこに置くか」を教えるときは、枠より点のほうが使いやすいことがあります。

In [ ]:
answer = ask("""Point to the flower and to the opening of the red bottle.
Return a JSON list like this: [{"point_2d": [x, y], "label": "target name"}]""", img)
print(answer)
points = parse_json(answer) or []
display(draw(img, points))

# ピクセル座標に直す (ロボットに渡すときはここから、カメラの校正を使って机の上の位置に変換する)
pil = load_image(img)
for p in points:
    x, y = to_px(*p["point_2d"], pil)
    print(f"{p['label']}: 正規化 {p['point_2d']} → ピクセル ({x:.0f}, {y:.0f})  画像 {pil.width}x{pil.height}")

## 4. 印を付けた物を説明させる (Describe Anything)

画像の中に番号の印が付いているとき、その印の物を 1 つずつ説明させられます。公式のお手本の画像には、すでに番号が描かれています。

In [ ]:
img = fetch(f"{FW}/reasoner/assets/describe_anything.png")
display(IPImage(img, width=480))
answer = ask('Please caption the notable attributes in the provided image. List and describe all marked subjects '
             'in the image with their categories and detailed captions using a json with keyword "subject_id", "category" and "caption".', img)
for s in parse_json(answer) or []:
    print(f"[{s.get('subject_id')}] {s.get('category')}: {s.get('caption')}\n")

## 5. 見つけた物に番号を振って、1 つずつ説明させる

2 で見つけた枠を画像に描き込み、番号付きの画像として渡し直します。
「どこにあるか」(グラウンディング) と「それは何か」(説明) を組み合わせる使い方です。

In [ ]:
img = fetch(f"{HF}/example_reasoning_input.png")
marked = load_image(img).copy()
d = ImageDraw.Draw(marked)
font = ImageFont.load_default(size=max(18, marked.width // 30))
for i, b in enumerate(boxes, 1):
    x1, y1 = to_px(*b["bbox_2d"][:2], marked)
    x2, y2 = to_px(*b["bbox_2d"][2:4], marked)
    d.rectangle([x1, y1, x2, y2], outline="red", width=3)
    d.text((x1 + 4, y1 + 2), str(i), fill="red", font=font, stroke_width=3, stroke_fill="white")
display(marked)
answer = ask('List and describe all marked subjects in the image with their categories and detailed captions '
             'using a json with keyword "subject_id", "category" and "caption".', marked)
for s in parse_json(answer) or []:
    print(f"[{s.get('subject_id')}] {s.get('category')}: {s.get('caption')}\n")

## 6. 自分の画像で試す

`target` に探したい物を英語で書いて実行してください。`mode` で枠か点かを選べます。
SO-101 などのロボットのカメラ画像で、つかみたい物を指させてみるのがおすすめです。

In [ ]:
# @title 自分の画像で試す (colab-only)
from google.colab import files

target = "every cup"  # @param {type:"string"}
mode = "bbox"  # @param ["bbox", "point"]

up = files.upload()
path = next(iter(up))
if mode == "bbox":
    q = f'Locate {target} and output the bounding boxes. Return a JSON list like this: [{{"bbox_2d": [x1, y1, x2, y2], "label": "name"}}]'
else:
    q = f'Point to {target}. Return a JSON list like this: [{{"point_2d": [x, y], "label": "name"}}]'
answer = ask(q, path)
print(answer)
display(draw(path, parse_json(answer) or []))

## うまくいかないとき

- **枠が少しずれる**: Edge (4B) は小さい物ほど枠がずれやすくなります。`seed` を変えて何回か聞き、平均を取ると安定します
- **JSON にならない**: 返してほしい形の例をそのまま質問文に入れると効きます (`[{"bbox_2d": [...], "label": "..."}]`)
- **似た物が複数あって取り違える**: 「左から 2 番目の赤いコップ」のように、位置や色で区別できる言い方にします
- **ピクセルに直すときの注意**: 0〜1000 は縦横それぞれの比率です。正方形でない画像でも `x / 1000 * 幅`、`y / 1000 * 高さ` で直します